# 02 · RQ1: forecasting, neighbor ablation, LOSO, SHAP

Baselines (persistence, seasonal-naive, Ridge) and LightGBM with 0 / 3 / 11 neighbor stations; Diebold-Mariano, Holm, Wilcoxon and block-bootstrap statistics; leave-one-station-out; TreeSHAP group shares. QUICK mode skips Optuna (default hyper-parameters). The TEST split is evaluated once in a full run; quick runs are demos and may be repeated.

*Estimated time: QUICK: 3-6 min; full run with Optuna: hours (resumable).*

In [ ]:
# --- parameters -------------------------------------------------------------------------
QUICK = True        # 2 stations, short run (< 5 min); set False for the full 12-station run
USE_DRIVE = False   # Colab only: mount Google Drive to keep results between sessions
RUN_OPTUNA = False  # hyper-parameter search (notebook 02 onwards)
SEEDS = [0]


In [ ]:
# --- setup (runs on Colab, skipped locally) ---------------------------------------------
import os
import subprocess
import sys
IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/furkantopcuu/airq-iot.git"
if IN_COLAB:
    if not os.path.exists("/content/airq-iot"):
        subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/airq-iot"], check=True)
    os.chdir("/content/airq-iot")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml]"], check=True)
    sys.path.insert(0, "/content/airq-iot/src")  # no restart needed after the editable install


In [ ]:
from airq.config import load_config
from airq.env import mount_drive_if_wanted
from airq.pipelines.colab import env_summary

mount_drive_if_wanted(USE_DRIVE)
cfg = load_config()
env_summary(cfg)


In [ ]:
from airq.pipelines.rq1 import run_v0
from airq.pipelines.rq1_final import evaluate_test, loso, shap_groups
from airq.pipelines.tuning import run_tuning

if RUN_OPTUNA:
    run_tuning(cfg, quick=QUICK, trials=20)  # TPE on validation only, resumable
else:
    run_v0(cfg, quick=QUICK)  # default hyper-parameters, validation only


In [ ]:
summary = evaluate_test(cfg, quick=QUICK, force=QUICK)  # full run: seen once
shap_groups(cfg, quick=QUICK)
loso(cfg, quick=QUICK, force=QUICK)
summary["by_horizon_best"]


In [ ]:
import pandas as pd
from IPython.display import Image, display
from airq.env import resolve_paths
from airq.pipelines.figures import rq1

paths = resolve_paths(cfg["paths"])
tables = paths["reports"] / "tables"
display(pd.read_csv(tables / "rq1_main.csv").round(2))
display(pd.read_csv(tables / "rq1_ablation.csv").round(2))
for p in rq1(cfg):
    if p.suffix == ".png":
        display(Image(filename=str(p)))


In [ ]:
# --- export: reports/ and models/ -> airq_artifacts_<date>.zip --------------------------
from airq.pipelines.colab import export_artifacts

zip_path = export_artifacts(cfg)
print("Wrote", zip_path.name)
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
# locally:  python -m scripts.import_artifacts <zip>   (also runs the manifest check)
